In [1]:
# Resolve the CDR dataset and the workspace bucket
cdr    <- trimws(tail(system("wb resource resolve --id=C2024Q3R9", intern = TRUE), 1))
bucket <- trimws(tail(system("wb resource resolve --id=rw-migration-aou-rw-6f1dba72", intern = TRUE), 1))

cdr     # expect something like "project-id.C2024Q3R9" (exactly one dot)
bucket  # expect "gs://..."

Sys.setenv(WORKSPACE_CDR = cdr, WORKSPACE_BUCKET = bucket)

[1] "wb-silky-artichoke-2408.C2024Q3R9"

[1] "gs://rw-migration-aou-rw-6f1dba72"

In [2]:
# ==== CELL 1   (setup)
library(data.table)
BASE <- "/home/jupyter/workspace/rw-migration-aou-rw-6f1dba72/workspaces"
MISS <- file.path(BASE, "temporaldigitalhealthphenotyping/final_missing")
source(file.path(MISS, "load_missingness.R")); load_missingness(MISS)
source(file.path(MISS, "R/stream_states.R")); source(file.path(MISS, "R/plots_stream_states.R"))
EXPORT_DATE <- "20261031"                                                   # the export of the people of the selection (as in notebook_cells.R)
L1_DIR      <- file.path(BASE, "result/final_missing/L1")
OUT_DIR     <- file.path(BASE, "result/final_missing/results")
OUT_STREAM  <- file.path(OUT_DIR, "stream"); dir.create(OUT_STREAM, recursive = TRUE, showWarnings = FALSE)
cfg <- make_config(list(export = list(date = EXPORT_DATE), overlap = list(policy = "exclude")))
st  <- st_config(list(seed = cfg$seed))             # settings of this analysis (cut-offs, gate limits, the list of versions without heart rate)
stopifnot("WORKSPACE_BUCKET / OWNER_EMAIL not set" = nzchar(cfg$export$bucket) && nzchar(cfg$export$owner))


In [3]:
# ==== CELL 2   (people, day tables and the devices of every day; the raw device export is read once)
inp <- load_inputs(cfg, L1_DIR); store <- inp$day_tables
persons <- copy(inp$persons); dp <- store$device_person()
stopifnot("run Cell 8b of notebook_cells.R first (device_person.rds is missing)" = !is.null(dp))
persons[dp, on = "person_id", device_family := i.device_family]
stopifnot("the person table needs cohort and age_group" = all(c("cohort", "age_group") %in% names(persons)))
raw_dev <- rbindlist(lapply(cfg$cohorts, function(co) read_export_files(export_path(cfg, co, "device"), c(device_version = "character"))), use.names = TRUE, fill = TRUE)
trk <- st_trackers(raw_dev, persons, cfg); rm(raw_dev)
lookup <- st_device_lookup(trk, st)                 # CHECK THIS TABLE: which versions are treated as having no heart-rate sensor (product knowledge, not data)
dir.create(file.path(OUT_STREAM, "tables"), showWarnings = FALSE); fwrite(lookup, file.path(OUT_STREAM, "tables", "stream_device_hr_lookup.tsv"), sep = "\t")
print(lookup, nrows = 60)


22:29:35 | reading 1 file(s) for insomnia_device_*.csv

22:29:39 | reading 1 file(s) for diabetes_device_*.csv

22:29:43 | reading 1 file(s) for healthy_device_*.csv

22:29:47 | reading 1 file(s) for pca_device_*.csv



        device_version n_rows n_people has_hr                         basis
                <char>  <int>    <int> <lgcl>                        <char>
 1:           Charge 5 158270     3521   TRUE assumed to measure heart rate
 2:            Versa 4  71088     2073   TRUE assumed to measure heart rate
 3:        MobileTrack  78049     1656  FALSE  listed as without heart rate
 4:            Versa 3 110486      774   TRUE assumed to measure heart rate
 5:            Versa 2  43202      407   TRUE assumed to measure heart rate
 6:           Charge 4  51027      392   TRUE assumed to measure heart rate
 7:          Inspire 2  35090      391   TRUE assumed to measure heart rate
 8:              Sense  47744      374   TRUE assumed to measure heart rate
 9:           Charge 2  23454      261   TRUE assumed to measure heart rate
10:               Luxe  15586      163   TRUE assumed to measure heart rate
11:            Sense 2   9178      149   TRUE assumed to measure heart rate
12:         

In [4]:
 # ==== CELL 3   (the person-day table: one row per person-day of every window; ~ a minute)
D <- st_build_days(store, persons, trk, st, cfg)
writeLines(st_methods_text(st), file.path(OUT_STREAM, "methods.md"))
print(D[, .N, by = .(cohort, reason)][order(cohort, reason)])


      cohort                 reason       N
      <fctr>                 <char>   <int>
 1: Insomnia               analysed  295102
 2: Insomnia device_unknown_removed   52064
 3: Insomnia      device_without_hr   13040
 4: Insomnia      stream_not_active   99165
 5: Diabetes               analysed  252184
 6: Diabetes device_unknown_removed   38517
 7: Diabetes      device_without_hr   11956
 8: Diabetes      stream_not_active   84792
 9:  Healthy               analysed 2150656
10:  Healthy device_unknown_removed  276143
11:  Healthy      device_without_hr   89021
12:  Healthy      stream_not_active  793046
13: Prostate               analysed   58338
14: Prostate device_unknown_removed     760
15: Prostate      device_without_hr    3210
16: Prostate      stream_not_active   11413


In [9]:
# ==== CELL 4   (PARTS A-C and the gate: data flow, the activity filter, patterns and nesting, date check, minutes check)
abc <- st_run_abc(D, st)
st_write_tables(abc, OUT_STREAM); st_save_figures(st_figures(abc, NULL, cfg, st), OUT_STREAM)
print(abc$flow[, .(cohort, step, days = n_days_at_step, share_pct = round(share_pct, 2), low = round(low, 2), high = round(high, 2))], nrows = 60)
print(abc$activity_filter[, .(cohort, step, n_days, n_events, share_pct = round(share_pct, 3))])
print(abc$combos[stage == "after the filter", .(cohort, label, category, share_pct = round(share_pct, 3))], nrows = 40)
print(abc$gate); print(abc$date_check)
st_require_gate(abc$gate)                           # stops here if the date check or the minutes check fails: fix the data preparation first


      cohort                                                               step
      <fctr>                                                             <char>
 1:  Healthy                                            all days in the windows
 2: Insomnia                                            all days in the windows
 3: Diabetes                                            all days in the windows
 4: Prostate                                            all days in the windows
 5:  Healthy                                       removed: a stream not active
 6: Insomnia                                       removed: a stream not active
 7: Diabetes                                       removed: a stream not active
 8: Prostate                                       removed: a stream not active
 9:  Healthy                                activity not active (not exclusive)
10: Insomnia                                activity not active (not exclusive)
11: Diabetes                            

In [7]:
# ==== CELL 4b   (ONLY IF THE GATE FAILED ON THE MINUTES: what the records above the length of the day look like; read-only, nothing is changed)
# ratio of the minutes to the length of the day (about 2 = rows counted twice), how concentrated in people, in which years
diag <- st_minutes_diagnosis(D); print(diag)
# the raw heart-rate export, one cohort at a time: the same person-days under three ways of combining the rows of a zone
tot <- rbindlist(lapply(cfg$cohorts, function(co) {
  raw <- read_export_files(export_path(cfg, co, "hr"), c(zone_name = "character"))
  r <- st_hr_raw_totals(raw[person_id %in% persons$person_id], cfg$etl$hr_zones); rm(raw); invisible(gc()); r
}))
print(st_hr_raw_check(tot))                       # the rule with ~0% above the length of the day is the right way to combine the rows


$by_ratio
   ratio_bin n_records
      <fctr>     <int>
1:   1.0-1.1      8100
2:   1.1-1.5       143
3:   1.5-1.9         1

$by_cohort
     cohort n_records n_people
     <fctr>     <int>    <int>
1:  Healthy      6065     1342
2: Insomnia      1101      250
3: Diabetes       853      199
4: Prostate       225       50

$concentration
   n_records n_people share_top10_people_pct share_top100_people_pct
       <int>    <int>                  <num>                   <num>
1:      8244     1841                10.4197                39.84716

$by_year
    year     N
   <int> <int>
1:  2020    27
2:  2021   410
3:  2022  2352
4:  2023  5455



22:35:10 | reading 1 file(s) for insomnia_hr.csv

22:35:19 | reading 1 file(s) for diabetes_hr.csv

22:35:26 | reading 1 file(s) for healthy_hr.csv

22:36:09 | reading 1 file(s) for pca_hr.csv



                                  rule n_person_days n_above share_above_pct
                                <char>         <int>   <int>           <num>
1: sum of all rows (what the ETL does)       3302312    8244       0.2496433
2: drop exact duplicate rows, then sum       3302312    8244       0.2496433
3:  largest row of each zone, then sum       3302312    8244       0.2496433
   person_days_with_duplicated_zone_rows
                                   <int>
1:                                     0
2:                                     0
3:                                     0


In [8]:
# ==== CELL 4c   (ONLY IF THE GATE FAILED ON THE MINUTES; choose ONE option from what Cell 4b showed, then run Cell 4 again; L1 and the other analyses are untouched)
# OPTION 1 - rows counted twice (Cell 4b: ratios near 2, one combining rule removes the problem): use the corrected minutes of that rule ("t_distinct" or "t_max")
# D <- st_use_hr_minutes(D, tot, "t_distinct", st)
# OPTION 2 - the minutes are slightly above the length of the day in the raw export itself (Cell 4b: ratios 1.0-1.1, no duplicated zone rows, spread over many people): the day was longer
#   than 24 hours (travel across time zones, daylight-saving dates outside the US rule). Records up to 10% above the length of the day are accepted by the check, and the heart-rate wear of Parts F and G is
#   capped at the length of the day. Records further above still count against the 0.1% limit.
st <- modifyList(st, list(minutes_tolerance = 0.1, cap_hr_minutes = TRUE))
D <- st_assign_reason(st_flags(D, st), st)
fwrite(st_hr_raw_check(tot), file.path(OUT_STREAM, "tables", "stream_hr_minutes_raw_check.tsv"), sep = "\t")
# then run Cell 4 again: the gate must pass before Cell 5


In [14]:
# ==== CELL 5   (PARTS D-G: what state 3 is, composition, what 'night missing' contains, partial wear)
dg <- st_run_dg(D, st)
st_write_tables(dg, OUT_STREAM); st_save_figures(st_figures(abc, dg, cfg, st), OUT_STREAM)
print(dg$state3_vs_state1); print(dg$night_habit); print(dg$probably_worn); print(dg$hr_wear_below)


      cohort                       day_type metric n_people         mean
      <fctr>                         <char> <char>    <int>        <num>
 1:  Healthy                        weekday  steps     1311 -1257.891306
 2:  Healthy                        weekend  steps      909  -839.868267
 3: Diabetes                        weekday  steps      192  -845.661023
 4: Diabetes                        weekend  steps      134  -732.817843
 5: Insomnia                        weekday  steps      227 -1168.151147
 6: Insomnia                        weekend  steps      146  -980.687971
 7: Prostate                        weekday  steps       29 -1203.478995
 8: Prostate                        weekend  steps       22  -669.008760
 9:  Healthy                        weekday   mvpa     1311    -4.800157
10:  Healthy                        weekend   mvpa      909    -3.488041
11: Diabetes                        weekday   mvpa      192    -1.604307
12: Diabetes                        weekend   mvpa 

In [15]:
# ==== CELL 6   (the check: the same analysis WITHOUT every day of unknown device; results in stream/no_unknown_device/)
st_chk <- modifyList(st, list(drop_unknown_days = TRUE))
D2 <- st_assign_reason(copy(D), st_chk)
chk <- st_run_variant(D2, st_chk, cfg, file.path(OUT_STREAM, "no_unknown_device"))
print(chk$abc$flow[step == "analysed days", .(cohort, days = n_days_at_step, share_pct = round(share_pct, 2))])


     cohort   days share_pct
     <fctr>  <int>     <num>
1:  Healthy 913374     27.60
2: Insomnia 136971     29.82
3: Diabetes 119409     30.82
4: Prostate  23216     31.49


In [10]:
# ==============================================================================
# STREAM MODEL (STREAM.md, "Stream model"): does a high-step day d predict the status of day d + 1?  Run AFTER Cell 4 passed (D = the person-day table, with the minutes option of Cell 4c if it was needed).
# Model A: one mixed logistic model per category (1-4 against 0); model B: one multinomial logit with person-clustered errors. Every fit is a checkpoint; all rows are used (nothing subsampled).
# ==============================================================================
# ==== CELL 7   (setup of the stream model, risk set, flow and event counts - look at the event counts BEFORE the models)
source(file.path(MISS, "R/stream_model.R")); source(file.path(MISS, "R/plots_stream_model.R"))
sc <- stm_config(list(seed = cfg$seed)); OUT_MODEL <- file.path(OUT_STREAM, "model"); dir.create(OUT_MODEL, showWarnings = FALSE)
CKPT <- file.path(OUT_DIR, cfg$checkpoint$dirname)
rs <- stm_risk_set(stm_days(D, sc), persons, cfg, sc); R <- rs$R
dkey <- list(export = EXPORT_DATE, n_rows = nrow(R), n_people = uniqueN(R$person_id))
print(rs$flow); print(rs$scales)
ev <- list(main = stm_event_counts(R, sc)); print(ev$main[group == "all"])
print(stm_cat4_composition(R))                      # which original patterns make up category 4


                                             step  n_rows n_dropped
                                           <char>   <int>     <int>
1: days d with all three streams (fully recorded) 2557622        NA
2:         >= 20 h of heart rate on d (and steps) 2505431     52191
3:                     >= 7 earlier baseline days 2450602     54829
4:            d is not the last day of the window 2444759      5843
5:                   d + 1 is not an excluded day 2440801      3958
        sd_w     mu_b      sd_b  n_rows n_people
       <num>    <num>     <num>   <int>    <int>
1: 0.6129778 8.753865 0.6518424 2440801     7799
    group  n_rows n_people category                           label  events
   <char>   <int>    <int>    <int>                          <char>   <int>
1:    all 2440801     7799        0                  Fully recorded 2332775
2:    all 2440801     7799        1                     All missing   13040
3:    all 2440801     7799        2                   Activity only    86

In [11]:
# ==== CELL 8   (main run: model A for categories 1-4 and the secondary split of category 3, model B; the longest cell - a checkpoint per model, so it can be re-run after a restart)
out_main <- stm_run_models(R, "main", cfg, sc, CKPT, dkey, secondary = TRUE, multinomial = TRUE)
res_main <- stm_collect(out_main, sc, "main")
print(res_main$tests[, .(model, or, or_low, or_high, p_wald, p_tost, p_tost_holm, equivalent, negative_control, n_events)])
print(res_main$contrasts); print(res_main$risk_diff)


22:39:06 | [skip] streammodel / main / c1 (checkpoint from 2026-11-11 05:18)

22:39:06 | [skip] streammodel / main / c2 (checkpoint from 2026-11-11 05:22)

22:39:07 | [skip] streammodel / main / c3 (checkpoint from 2026-11-11 05:25)

22:39:07 | [skip] streammodel / main / c4 (checkpoint from 2026-11-11 05:31)

22:39:07 | [skip] streammodel / main / c3_probably_worn_20 (checkpoint from 2026-11-11 05:35)

22:39:07 | [skip] streammodel / main / c3_probably_not_worn_20 (checkpoint from 2026-11-11 05:41)

22:39:07 | [skip] streammodel / main / c3_probably_worn_22 (checkpoint from 2026-11-11 05:44)

22:39:07 | [skip] streammodel / main / c3_probably_not_worn_22 (checkpoint from 2026-11-11 05:51)

22:39:07 | [run] streammodel / main / multinomial  (session 3.8 GB, free 21.7 GB)



Key: <model>
                     model        or    or_low   or_high        p_wald
                    <char>     <num>     <num>     <num>         <num>
1:                      c1 0.8034451 0.7926306 0.8144073 7.089257e-220
2:                      c2 0.8445308 0.8302103 0.8590983  1.513118e-83
3:                      c3 0.8261007 0.8211244 0.8311072  0.000000e+00
4: c3_probably_not_worn_20 0.9876068 0.9593741 1.0166703  3.993730e-01
5: c3_probably_not_worn_22 0.9761388 0.9510504 1.0018889  6.907291e-02
6:     c3_probably_worn_20 0.8178876 0.8128192 0.8229877  0.000000e+00
7:     c3_probably_worn_22 0.8162964 0.8112040 0.8214208  0.000000e+00
8:                      c4 0.9330956 0.9193363 0.9470609  6.467314e-20
      p_tost p_tost_holm equivalent negative_control n_events
       <num>       <num>     <lgcl>           <char>    <int>
1: 1.0000000          NA      FALSE             <NA>    13040
2: 1.0000000           1      FALSE             <NA>     8694
3: 1.0000000           1     

In [12]:
# ==== CELL 9   (sensitivity: more than 16 h / 22 h of heart-rate wear on d, first 14 days excluded, first transition only; model A only)
runs <- list(wear_ge16h = stm_risk_set(stm_days(D, stm_config(list(wear_hours = 16, seed = cfg$seed))), persons, cfg, sc)$R,
             wear_ge22h = stm_risk_set(stm_days(D, stm_config(list(wear_hours = 22, seed = cfg$seed))), persons, cfg, sc)$R,
             first14 = stm_variant_rows(R, "first14", sc), first_transition = stm_variant_rows(R, "first_transition", sc))
res_var <- lapply(names(runs), function(v) { ev[[v]] <<- stm_event_counts(runs[[v]], sc)
  stm_collect(stm_run_models(runs[[v]], v, cfg, sc, CKPT, c(dkey, list(n = nrow(runs[[v]]))), secondary = FALSE, multinomial = FALSE), sc, v) })
names(res_var) <- names(runs)


22:41:50 | [skip] streammodel / wear_ge16h / c1 (checkpoint from 2026-11-11 05:56)

22:41:51 | [skip] streammodel / wear_ge16h / c2 (checkpoint from 2026-11-11 06:00)

22:41:51 | [skip] streammodel / wear_ge16h / c3 (checkpoint from 2026-11-11 06:03)

22:41:51 | [skip] streammodel / wear_ge16h / c4 (checkpoint from 2026-11-11 06:08)

22:41:51 | [skip] streammodel / wear_ge22h / c1 (checkpoint from 2026-11-11 06:12)

22:41:51 | [skip] streammodel / wear_ge22h / c2 (checkpoint from 2026-11-11 06:16)

22:41:51 | [skip] streammodel / wear_ge22h / c3 (checkpoint from 2026-11-11 06:19)

22:41:51 | [skip] streammodel / wear_ge22h / c4 (checkpoint from 2026-11-11 06:25)

22:41:52 | [skip] streammodel / first14 / c1 (checkpoint from 2026-11-11 06:29)

22:41:52 | [skip] streammodel / first14 / c2 (checkpoint from 2026-11-11 06:32)

22:41:52 | [skip] streammodel / first14 / c3 (checkpoint from 2026-11-11 06:36)

22:41:52 | [skip] streammodel / first14 / c4 (checkpoint from 2026-11-11 06:41)

22:4

In [13]:
# ==== CELL 10   (cohort-specific fits, only for cohorts with enough category-1 events; checks of the engine)
big <- ev$main[category == 1L & group != "all" & events >= sc$min_events_effect, group]
res_coh <- lapply(big, function(co) stm_collect(stm_run_models(R[cohort == co], paste0("cohort_", co), cfg, sc, CKPT, c(dkey, list(co = co)), secondary = FALSE, multinomial = FALSE), sc, paste0("cohort_", co)))
checks <- list(clogit = stm_clogit(R, sc), engine = stm_verify_engine(R, cfg, sc)); print(checks)


22:41:53 | [skip] streammodel / cohort_Healthy / c1 (checkpoint from 2026-11-11 06:52)

22:41:53 | [skip] streammodel / cohort_Healthy / c2 (checkpoint from 2026-11-11 06:55)

22:41:53 | [skip] streammodel / cohort_Healthy / c3 (checkpoint from 2026-11-11 06:58)

22:41:53 | [skip] streammodel / cohort_Healthy / c4 (checkpoint from 2026-11-11 07:03)

22:41:53 | [skip] streammodel / cohort_Insomnia / c1 (checkpoint from 2026-11-11 07:04)

22:41:53 | [skip] streammodel / cohort_Insomnia / c2 (checkpoint from 2026-11-11 07:06)

22:41:53 | [skip] streammodel / cohort_Insomnia / c3 (checkpoint from 2026-11-11 07:08)

22:41:53 | [skip] streammodel / cohort_Insomnia / c4 (checkpoint from 2026-11-11 07:09)



$clogit
       model  n_rows n_people n_events   estimate          se        or
      <char>   <int>    <int>    <int>      <num>       <num>     <num>
1: clogit_c1 1284731     2438    13040 -0.2105645 0.006457469 0.8101268
     or_low   or_high
      <num>     <num>
1: 0.799938 0.8204455

$engine
   n_people n_events w_lme4 w_glmmTMB difference
      <int>    <int>  <num>     <num>      <num>
1:     1500     2526     NA        NA         NA



In [14]:
# ==== CELL 11   (tables and figures)
all_res <- c(list(main = res_main), res_var, setNames(res_coh, paste0("cohort_", big)))
bindr <- function(nm) rbindlist(lapply(all_res, `[[`, nm), fill = TRUE)
res <- list(info = bindr("info"), coefficients = bindr("coefficients"), tests = bindr("tests"), multinomial_coefficients = res_main$multinomial_coefficients,
            contrasts = res_main$contrasts, risk_curve = res_main$risk_curve, risk_diff = res_main$risk_diff)
ec <- rbindlist(lapply(names(ev), function(v) ev[[v]][, variant := v]))
figs <- stm_figures(res, stm_deciles(R, sc), sc, ec)
stm_write(c(res, list(event_counts = ec, flow = rs$flow, scales = rs$scales, category4_composition = stm_cat4_composition(R), checks_clogit = checks$clogit, checks_engine = checks$engine)), figs, OUT_MODEL)
